# Compute distortion matrix based on a checkerboard video

Code based on https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html

In [ ]:
import cv2 as cv
import numpy as np
import pathlib

In [ ]:
# input video
checkerboard_video_path = pathlib.Path("../data/checkerboard_vid/checkerboard_test_clip.mp4")
checkerboard_dims = (6, 9)

# number of frames to use
n_frames = 30

# output for extracted frames
frames_path = pathlib.Path("../data/checkerboard_frames/")
frames_path.mkdir(exist_ok=True, parents=True)

# show checkerboard detections
show_checkerboards = False

In [ ]:
cap = cv.VideoCapture(checkerboard_video_path.as_posix())
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

# select and save random frames
frames = sorted(np.random.permutation(total_frames)[:n_frames])
for frame in frames:
    cap.set(cv.CAP_PROP_POS_FRAMES, frame)
    res, img = cap.read()
    if res:
        out_path = frames_path / f"frame_{frame}_raw.png"
        cv.imwrite(out_path.as_posix(), img)
    else:
        print(f"Could not extract frame {frame} (out of {total_frames}), skipping.")

In [ ]:
# termination criteria for checkerboard corner refinement
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((np.prod(checkerboard_dims), 3), np.float32)
objp[:,:2] = np.mgrid[0:checkerboard_dims[0],0:checkerboard_dims[1]].T.reshape(-1,2)
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

frames = frames_path.glob("*.png")
 
for frame in frames:
    img = cv.imread(frame.as_posix())
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, checkerboard_dims, None)
 
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
 
        corners2 = cv.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)
 
        if show_checkerboards:
            # Draw and display the corners
            cv.drawChessboardCorners(img, checkerboard_dims, corners2, ret)
            cv.imshow('img', img)
            cv.waitKey(500)
    else:
        print(f"No checkerboard found in {frame}, skipped.")

cv.destroyAllWindows()

In [ ]:
# Compute calibration parameters
ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
# TODO save these as .npy

In [ ]:
# Compute calibration error
total_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
    error = cv.norm(imgpoints[i], imgpoints2, cv.NORM_L2)/len(imgpoints2)
    total_error += error

print(f"Mean error: {total_error/len(objpoints)}")

In [ ]:
# Undistort images
input_path = pathlib.Path("../data/checkerboard_frames/")
output_path = pathlib.Path("../data/checkerboard_frames/")

frames = input_path.glob("*raw.png")

for i, frame in enumerate(frames):
    img = cv.imread(frame.as_posix())

    if i == 0:
        # refine the camera matrix, compute only once
        # TODO save these as .npy
        h,  w = img.shape[:2]
        newcameramtx, roi = cv.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))
        mapx, mapy = cv.initUndistortRectifyMap(mtx, dist, None, newcameramtx, (w,h), 5)

    # undistort
    dst = cv.remap(img, mapx, mapy, cv.INTER_LINEAR)
    
    # crop the image
    x, y, w, h = roi
    dst = dst[y:y+h, x:x+w]

    # write result
    frame_number = frame.name.rsplit(sep="_")[1]
    out_path = output_path / f"frame_{frame_number}_cal.png"
    cv.imwrite(out_path.as_posix(), dst)